# 勇者傳說 — Audio Generator (T4 GPU, Self-Contained)

Generate BGM and SFX using **MusicGen Large** (3.3B) on Colab T4.

| Model | VRAM | Speed | Quality |
|-------|------|-------|---------|
| MusicGen Large fp16 | ~6 GB | ~10s/30s | Best |
| MusicGen Medium fp16 | ~3 GB | ~6s/30s | Good |
| MusicGen Small fp32 | ~1.5 GB | ~60s/30s (CPU ok) | Basic |

**Segment chaining**: 30s 片段串接 + crossfade → 60-90s loopable BGM  
**Self-contained**: No repo clone needed.  
**Output**: Google Drive `/MyDrive/ai-rpg-game/outputs/audio/`  
**Resume**: 中斷後重跑自動跳過已完成的

## 1. Setup

In [ ]:
!pip install -q transformers accelerate scipy
!apt-get -qq install ffmpeg

import gc, json, os, shutil, subprocess, time
from pathlib import Path
import numpy as np

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

GDRIVE_BASE = Path('/content/drive/MyDrive/ai-rpg-game')
OUTPUT_BASE = GDRIVE_BASE / 'outputs' if ON_COLAB else Path('outputs')
MODELS_DIR = GDRIVE_BASE / 'models' if ON_COLAB else Path('models')

for d in [OUTPUT_BASE / 'audio' / 'bgm', OUTPUT_BASE / 'audio' / 'sfx', MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DTYPE = torch.float16
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
else:
    DEVICE = 'cpu'
    DTYPE = torch.float32
    print('No GPU -- will use MusicGen Small on CPU (slow)')

print(f'Output: {OUTPUT_BASE / "audio"}')

## 2. Utilities

In [ ]:
class ProgressTracker:
    def __init__(self, task_name, output_dir):
        self.file = Path(output_dir) / f'_progress_{task_name}.json'
        self.completed = set()
        self.start_time = time.time()
        if self.file.exists():
            try:
                data = json.loads(self.file.read_text())
                self.completed = set(data.get('completed', []))
                print(f'[resume] {len(self.completed)} items already done')
            except Exception:
                pass

    def is_done(self, name): return name in self.completed

    def mark_done(self, name):
        self.completed.add(name)
        self.file.parent.mkdir(parents=True, exist_ok=True)
        self.file.write_text(json.dumps({
            'completed': sorted(self.completed),
            'count': len(self.completed),
            'last_updated': time.strftime('%Y-%m-%d %H:%M:%S'),
        }, indent=2))

    def summary(self, total):
        done = len(self.completed)
        elapsed = time.time() - self.start_time
        if done > 0:
            remaining = (total - done) * (elapsed / done)
            return f'{done}/{total} done, ~{remaining/60:.0f} min remaining'
        return f'0/{total} done'


def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Audio Prompt Data (Embedded)

In [ ]:
AUDIO_PROMPTS = {
    "_meta": {
        "style_prefix": "fantasy RPG game music, medieval, orchestral",
        "sfx_prefix": "game sound effect"
    },
    "bgm": [
        {"name": "title", "duration": 30, "prompt": "epic fantasy RPG opening theme, harp and strings, majestic, adventurous"},
        {"name": "town", "duration": 30, "prompt": "cozy RPG town music, acoustic guitar, flute, warm and peaceful village"},
        {"name": "field", "duration": 30, "prompt": "adventure exploration music, strings and woodwinds, open world, hopeful"},
        {"name": "battle", "duration": 30, "prompt": "aggressive fast-paced battle music, driving heavy drums 160bpm, urgent brass stabs, relentless percussion, intense combat, adrenaline rush, heroic tension"},
        {"name": "boss", "duration": 30, "prompt": "dark epic boss battle theme, dramatic choir, heavy percussion, menacing"},
        {"name": "victory", "duration": 10, "prompt": "triumphant victory fanfare, brass and strings, celebration, short"},
        {"name": "gameover", "duration": 15, "prompt": "sad game over music, slow piano, melancholy, somber"},
        {"name": "shop", "duration": 20, "prompt": "cheerful medieval shop music, lute and recorder, trading, lighthearted"},
        {"name": "town_r1", "duration": 60, "prompt": "peaceful pastoral village music, gentle acoustic guitar and flute, birdsong ambience, warm sunshine meadow feel, relaxing countryside"},
        {"name": "town_r2", "duration": 60, "prompt": "mystical forest village music, ethereal harp and pan flute, gentle wind chimes, enchanted woodland, elvish serenity"},
        {"name": "town_r3", "duration": 60, "prompt": "tropical island port town music, steel drums and ukulele, ocean waves ambience, Caribbean calypso feel, laid-back seaside"},
        {"name": "town_r4", "duration": 60, "prompt": "savanna tribal village music, djembe drums and kalimba, warm rhythmic patterns, African-inspired percussion, sunset grasslands"},
        {"name": "town_r5", "duration": 60, "prompt": "frozen northern village music, haunting Nordic fiddle and kantele, wind howling softly, cozy fireplace warmth against winter cold"},
        {"name": "town_r6", "duration": 60, "prompt": "mountain fortress town music, military snare and brass fanfare, noble and proud, stone castle grandeur, highland pipes hint"},
        {"name": "town_r7", "duration": 60, "prompt": "eerie swamp village music, muted strings and oboe, creaking wood, mysterious fog, haunting minor key melody"},
        {"name": "town_r8", "duration": 60, "prompt": "eastern temple village music, koto and shakuhachi flute, bamboo percussion, zen garden tranquility, Japanese-inspired"},
        {"name": "town_r9", "duration": 60, "prompt": "floating sky city music, celestial bells and strings, airy ethereal pads, heavenly choir whispers, serene above the clouds"},
        {"name": "town_r10", "duration": 60, "prompt": "volcanic forge town music, heavy anvil percussion, deep bass rumble, industrial hammering rhythm, determined smithing energy"},
        {"name": "town_r11", "duration": 60, "prompt": "arcane academy town music, mysterious celesta and harpsichord, magical twinkling, scholarly waltz, library ambience"},
        {"name": "town_r12", "duration": 60, "prompt": "dark castle stronghold music, ominous pipe organ and choir, deep reverberant bells, gothic cathedral atmosphere, foreboding"},
        {"name": "boss_r1", "duration": 60, "prompt": "corrupted paladin boss battle theme, dramatic orchestral with heavy choir, clashing swords percussion, noble melody twisted dark"},
        {"name": "boss_r6", "duration": 60, "prompt": "mountain siege boss battle theme, war drums and brass, military march turned chaotic, desperate combat"},
        {"name": "boss_r12", "duration": 60, "prompt": "final demon lord boss battle theme, apocalyptic orchestra and choir, pipe organ, fastest tempo, ultimate confrontation, all instruments, most intense"}
    ],
    "sfx": [
        {"name": "sfx_select", "duration": 1, "prompt": "bright single xylophone note ping, crisp high pitched bell chime, staccato, clean digital tone"},
        {"name": "sfx_cancel", "duration": 1, "prompt": "two quick descending piano notes, minor interval, soft muted pluck, short falling tone"},
        {"name": "sfx_hit", "duration": 1, "prompt": "sharp percussive snare hit with metallic crash cymbal, aggressive attack impact, punchy drum strike"},
        {"name": "sfx_magic", "duration": 2, "prompt": "ethereal ascending harp glissando with celesta sparkle, shimmering crystal chimes, mystical fairy dust"},
        {"name": "sfx_heal", "duration": 3, "prompt": "healing spell sound effect, gentle chime, restoration, warm glow"},
        {"name": "sfx_footstep", "duration": 1, "prompt": "single footstep on stone floor, soft leather boot step, quiet walking sound"},
        {"name": "sfx_door", "duration": 2, "prompt": "heavy wooden door opening with creak, medieval oak door hinge, old rusty hinge squeak"},
        {"name": "sfx_chest", "duration": 2, "prompt": "treasure chest opening sound, wooden lid creak then magical sparkle chime, discovery jingle"},
        {"name": "sfx_levelup", "duration": 3, "prompt": "triumphant level up fanfare, ascending arpeggio, bright brass and bells, achievement celebration, short jingle"},
        {"name": "sfx_equip", "duration": 1, "prompt": "metallic armor equip sound, chain mail clink, weapon sheathe, gear changing metallic"},
        {"name": "sfx_coin", "duration": 1, "prompt": "gold coins dropping and clinking, metallic jingling, treasure pickup, bright coin sound"},
        {"name": "sfx_save", "duration": 2, "prompt": "save point activation, gentle crystalline chime ascending, reassuring holy bell, checkpoint reached"},
        {"name": "sfx_flee", "duration": 2, "prompt": "running away quick footsteps, panicked retreat, whooshing escape, diminishing patter"},
        {"name": "sfx_critical", "duration": 2, "prompt": "powerful critical hit impact, explosive bass slam with metallic ring, devastating heavy blow, screen-shaking impact"},
        {"name": "sfx_encounter", "duration": 2, "prompt": "battle encounter alert, dramatic two-note brass stab, danger warning, sudden tension, enemy spotted"}
    ]
}

print(f"BGM: {len(AUDIO_PROMPTS['bgm'])} tracks")
print(f"SFX: {len(AUDIO_PROMPTS['sfx'])} effects")

## 4. Load MusicGen Pipeline

Change `MODEL_SIZE` to `'medium'` or `'small'` if T4 has VRAM issues (unlikely with Large at ~6GB).

In [ ]:
from transformers import pipeline as hf_pipeline

MODEL_SIZE = 'large'  # 'large', 'medium', 'small'

hub_ids = {
    'large': 'facebook/musicgen-large',
    'medium': 'facebook/musicgen-medium',
    'small': 'facebook/musicgen-small',
}
hub_id = hub_ids[MODEL_SIZE]

# Check Drive cache
cache_dir = MODELS_DIR / f'musicgen-{MODEL_SIZE}'
model_path = str(cache_dir) if cache_dir.exists() else hub_id
if model_path == hub_id:
    print(f'[download] {hub_id} (first run will download ~6GB)')
else:
    print(f'[cache] {cache_dir}')

print(f'[pipeline] MusicGen {MODEL_SIZE.capitalize()} ({DEVICE}, {DTYPE})...')
t0 = time.time()

audio_pipe = hf_pipeline(
    'text-to-audio',
    model=model_path,
    device=DEVICE,
    torch_dtype=DTYPE,
)

print(f'[pipeline] Ready in {time.time()-t0:.1f}s')

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated(0) / 1024**3
    props = torch.cuda.get_device_properties(0)
    total = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1024**3
    print(f'[vram] {used:.1f}/{total:.1f} GB')

## 5. Audio Generation + Segment Chaining Functions

In [ ]:
SAMPLE_RATE = 32000
TOKENS_PER_SECOND = 50
MAX_TOKENS = 1503  # ~30s per segment


def generate_segment(prompt, max_tokens):
    result = audio_pipe(prompt, forward_params={'max_new_tokens': max_tokens})
    return np.squeeze(result['audio'])


def crossfade(a, b, overlap):
    overlap = min(overlap, len(a), len(b))
    if overlap <= 0:
        return np.concatenate([a, b])
    fade_out = np.linspace(1.0, 0.0, overlap)
    fade_in = np.linspace(0.0, 1.0, overlap)
    mixed = a[-overlap:] * fade_out + b[:overlap] * fade_in
    return np.concatenate([a[:-overlap], mixed, b[overlap:]])


def make_loop_seamless(audio, fade_samples):
    fade = min(fade_samples, len(audio) // 4)
    if fade <= 0:
        return audio
    result = audio.copy()
    fade_out = np.linspace(1.0, 0.0, fade)
    fade_in = np.linspace(0.0, 1.0, fade)
    result[:fade] = audio[:fade] * fade_in + audio[-fade:] * fade_out
    return result[:-fade]


def generate_chained_audio(prompt, total_duration, segment_duration=30,
                           crossfade_seconds=3.0, loop=True):
    """Chain multiple 30s segments for longer tracks."""
    segments = []
    remaining = total_duration
    idx = 0

    while remaining > 0:
        dur = min(remaining, segment_duration)
        tokens = min(dur * TOKENS_PER_SECOND, MAX_TOKENS)
        seg_prompt = f'{prompt}, continuation' if idx > 0 else prompt
        print(f'    [segment {idx+1}] {dur}s ({tokens} tokens)')
        segments.append(generate_segment(seg_prompt, tokens))
        remaining -= dur
        idx += 1

    overlap = int(crossfade_seconds * SAMPLE_RATE)
    combined = segments[0]
    for seg in segments[1:]:
        combined = crossfade(combined, seg, overlap)

    if loop and total_duration >= 30:
        combined = make_loop_seamless(combined, overlap)

    return combined


def normalize_loudness(audio, target_db=-14.0):
    rms = np.sqrt(np.mean(audio ** 2))
    if rms < 1e-8:
        return audio
    target_rms = 10 ** (target_db / 20.0)
    return audio * (target_rms / rms)


def convert_to_ogg(wav_path):
    if not shutil.which('ffmpeg'):
        return None
    ogg_path = wav_path.with_suffix('.ogg')
    try:
        subprocess.run(
            ['ffmpeg', '-y', '-i', str(wav_path),
             '-c:a', 'libvorbis', '-q:a', '4', str(ogg_path)],
            capture_output=True, check=True,
        )
        print(f'        ogg: {ogg_path.name} ({ogg_path.stat().st_size / 1024:.0f} KB)')
        return ogg_path
    except Exception:
        return None


def generate_audio_entry(entry, meta, category, tracker, chain_segments=True):
    """Generate a single audio entry. Returns True if generated."""
    import scipy.io.wavfile

    name = entry['name']
    duration = entry['duration']
    subdir = 'bgm' if category == 'bgm' else 'sfx'
    out_dir = OUTPUT_BASE / 'audio' / subdir
    wav_path = out_dir / f'{name}.wav'
    ogg_path = out_dir / f'{name}.ogg'

    if tracker.is_done(name) and (ogg_path.exists() or wav_path.exists()):
        print(f'  [skip] {name}')
        return False

    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = meta.get('sfx_prefix') if category == 'sfx' else meta.get('style_prefix', '')
    prompt = f"{prefix}, {entry['prompt']}"
    target_lufs = -14.0 if category == 'bgm' else -10.0

    print(f'  [gen] {name} ({duration}s)')
    print(f'        prompt: {prompt[:80]}...')
    t0 = time.time()

    if chain_segments and category == 'bgm' and duration > 30:
        print(f'        chaining: {duration}s in ~30s segments')
        audio = generate_chained_audio(prompt, duration)
    else:
        tokens = min(duration * TOKENS_PER_SECOND, MAX_TOKENS)
        print(f'        tokens: {tokens} (~{tokens // TOKENS_PER_SECOND}s)')
        audio = generate_segment(prompt, tokens)

    audio = normalize_loudness(audio, target_lufs)
    audio = np.clip(audio, -1.0, 1.0)

    scipy.io.wavfile.write(str(wav_path), SAMPLE_RATE, (audio * 32767).astype(np.int16))

    elapsed = time.time() - t0
    kb = wav_path.stat().st_size / 1024
    print(f'        saved: {wav_path.name} ({kb:.0f} KB, {elapsed:.1f}s)')

    convert_to_ogg(wav_path)
    tracker.mark_done(name)
    free_vram()
    return True

## 6. Test: Generate 1 BGM + 1 SFX

Quick quality check before full batch.

In [ ]:
from IPython.display import Audio, display

meta = AUDIO_PROMPTS['_meta']
tracker = ProgressTracker('audio_test', OUTPUT_BASE)

# Test BGM
generate_audio_entry(AUDIO_PROMPTS['bgm'][0], meta, 'bgm', tracker, chain_segments=False)

# Test SFX
generate_audio_entry(AUDIO_PROMPTS['sfx'][0], meta, 'sfx', tracker, chain_segments=False)

# Listen
for cat in ['bgm', 'sfx']:
    cat_dir = OUTPUT_BASE / 'audio' / cat
    if not cat_dir.exists():
        continue
    for f in sorted(cat_dir.glob('*.wav'))[:1]:
        print(f'\n{cat.upper()}: {f.name}')
        display(Audio(str(f)))

## 7. Generate All BGM (23 tracks)

8 base + 12 region-specific + 3 boss variants.  
Tracks >30s use segment chaining (2x30s + crossfade).  
Estimated: ~15-20 minutes on T4.

In [ ]:
category = 'bgm'
entries = AUDIO_PROMPTS[category]
meta = AUDIO_PROMPTS['_meta']

tracker = ProgressTracker(f'audio_{category}', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'\nBGM: {len(entries)} total, {remaining} to generate')

for i, entry in enumerate(entries):
    print(f'\n[{i+1}/{len(entries)}] {tracker.summary(len(entries))}')
    generate_audio_entry(entry, meta, category, tracker, chain_segments=True)

print(f'\nDone!')

## 8. Generate All SFX (15 effects)

In [ ]:
category = 'sfx'
entries = AUDIO_PROMPTS[category]
meta = AUDIO_PROMPTS['_meta']

tracker = ProgressTracker(f'audio_{category}', OUTPUT_BASE)
remaining = sum(1 for e in entries if not tracker.is_done(e['name']))
print(f'\nSFX: {len(entries)} total, {remaining} to generate')

for i, entry in enumerate(entries):
    print(f'\n[{i+1}/{len(entries)}] {tracker.summary(len(entries))}')
    generate_audio_entry(entry, meta, category, tracker, chain_segments=False)

print(f'\nDone!')

## 9. Preview All Audio

In [ ]:
import matplotlib.pyplot as plt
import scipy.io.wavfile

audio_dir = OUTPUT_BASE / 'audio'

for cat in ['bgm', 'sfx']:
    cat_dir = audio_dir / cat
    if not cat_dir.exists():
        continue
    wavs = sorted(cat_dir.glob('*.wav'))
    if not wavs:
        continue

    print(f'\n=== {cat.upper()} ({len(wavs)} tracks) ===')
    for wav_path in wavs:
        sr, data = scipy.io.wavfile.read(str(wav_path))
        duration = len(data) / sr
        print(f'\n{wav_path.stem} ({duration:.1f}s):')
        display(Audio(str(wav_path)))

        fig, ax = plt.subplots(figsize=(10, 1.5))
        t = np.linspace(0, duration, len(data))
        ax.plot(t, data, linewidth=0.3, color='steelblue')
        ax.set_xlim(0, duration)
        ax.set_ylabel('Amp')
        ax.set_title(wav_path.stem, fontsize=10)
        plt.tight_layout()
        plt.show()

## 10. Generate Manifest + Download

In [ ]:
# Generate audio manifest
audio_dir = OUTPUT_BASE / 'audio'
manifest = {'bgm': [], 'sfx': []}

for category in ['bgm', 'sfx']:
    cat_dir = audio_dir / category
    if not cat_dir.exists():
        continue
    for f in sorted(cat_dir.iterdir()):
        if f.suffix in ('.ogg', '.wav'):
            name = f.stem
            ogg_exists = (cat_dir / f'{name}.ogg').exists()
            ext = 'ogg' if ogg_exists else 'wav'
            if not any(e['name'] == name for e in manifest[category]):
                manifest[category].append({
                    'name': name,
                    'file': f'{category}/{name}.{ext}',
                })

manifest_path = audio_dir / 'manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f"BGM: {len(manifest['bgm'])} | SFX: {len(manifest['sfx'])}")

# Zip + download
zip_path = '/content/audio_output'
shutil.make_archive(zip_path, 'zip', str(audio_dir))
print(f'\nZip: {os.path.getsize(zip_path + ".zip") / 1024 / 1024:.1f} MB')

if ON_COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
else:
    print(f'Download: {zip_path}.zip')